# Case Study: Scientific Data Sanitization

Migrating scientific datasets from legacy formats into strictly typed Lakehouse architectures (e.g., Apache Iceberg) introduces significant validation challenges. 

The Automated Tropical Cyclone Forecasting (ATCF) format is a global meteorological standard. Raw ATCF text exports contain domain-specific artifacts and historical, physically impossible data anomalies, such as an **outer closed isobar pressure recorded as lower than the minimum central pressure**.

This case study demonstrates how to construct a data sanitization pipeline in Polars to resolve formatting and physical anomalies, followed by utilizing Veridelta to mathematically prove the pipeline introduced no unintended regressions.

## 1. The Legacy Dataset

The simulated ATCF "Best Track" dataset exhibits several structural and scientific anomalies requiring intervention:
* `basin`: Uses outdated 2-letter codes with inconsistent casing (`AL`, `al`).
* `lat`: Contains embedded hemisphere markers (`N`), coercing the column to a String type.
* `rad`: Contains a known legacy integer bug (`35` instead of the specified `34`).
* `outer_pressure_mb`: On storm `EP012026`, the outer pressure (`990`) is lower than the central `min_pressure_mb` (`1005`). This violates physical laws.

In [ ]:
import polars as pl

from veridelta.engine import DiffEngine
from veridelta.models import DiffConfig, DiffRule

# Source: Raw Legacy Export
legacy_lf = pl.LazyFrame(
    {
        "storm_id": ["AL092026", "AL092026", "EP012026"],
        "timestamp": ["2026090112", "2026090118", "2026051506"],
        "basin": ["AL", "al", "EP"],
        "lat": ["28.5N", "29.1N", "14.2N"],
        "rad": [35, 35, 50],
        "min_pressure_mb": [940, 935, 1005],
        "outer_pressure_mb": [1013, 1010, 990],  # Row 3 anomaly: 990 < 1005
    }
)

print("SYSTEM: Legacy dataset loaded into computation graph.")

## 2. The Data Engineering Pipeline

A standard Polars ETL pipeline sanitizes the formatting, resolves the integer bug, and enforces physical limits by coercing impossible pressure readings to `Null`.

In [ ]:
# pyright: reportUnknownMemberType=false


def migrate_to_lakehouse(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Execute sanitization and enforce strictly-typed schema."""
    return lf.with_columns(
        [
            # Standardize casing and apply 3-letter basin codes
            pl.col("basin").str.to_uppercase().replace({"AL": "ATL", "EP": "EPAC"}),
            # Strip hemisphere markers and cast to Float64
            pl.col("lat").str.replace("N", "").cast(pl.Float64),
            # Resolve legacy integer bug
            pl.col("rad").replace({35: 34}).cast(pl.Float64),  # type: ignore
            # Enforce Physical Constraints: Coerce invalid pressure gradients to Null
            pl.when(pl.col("outer_pressure_mb") < pl.col("min_pressure_mb"))
            .then(None)
            .otherwise(pl.col("outer_pressure_mb"))
            .cast(pl.Float64)
            .alias("outer_pressure_mb"),
        ]
    )


# Target: Cleaned Lakehouse Table
modern_lf = migrate_to_lakehouse(legacy_lf)

print("SYSTEM: ETL Pipeline transformation defined.")

## 3. Semantic Validation with Veridelta

Standard row-by-row equality checks evaluate to `False` across the dataset due to the intentional formatting and type modifications. 

To isolate true data regressions, we define a declarative Veridelta `DiffConfig`. This executable contract bridges the known, acceptable formatting gaps.

In [ ]:
validation_contract = DiffConfig(
    primary_keys=["storm_id", "timestamp"],
    output_path="./audit_logs",
    output_format="parquet",
    rules=[
        DiffRule(
            column_names=["basin"], case_insensitive=True, value_map={"AL": "ATL", "EP": "EPAC"}
        ),
        DiffRule(column_names=["lat"], regex_replace={"N|E": ""}, cast_to="Float64"),
        DiffRule(column_names=["rad"], value_map={35: 34}),
        DiffRule(column_names=["min_pressure_mb", "outer_pressure_mb"], cast_to="Float64"),
    ],
)

engine = DiffEngine(validation_contract)
final_summary = engine.compare_lazyframes(legacy_lf, modern_lf)

print(final_summary.report_summary)

# Output:
# Veridelta Execution Summary
# ===========================
# Status:        FAILED
# Match Rate:    66.67%
# Source Rows:   3
# Target Rows:   3
# Volume Shift:  +0 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         0
# Removed:       0
# Changed:       1
# Total Issues:  1
#
# Top Column-Level Drifts:
# ---------------------------
# - outer_pressure_mb: 1 mismatches

## 4. Auditing Intentional Business Logic

Veridelta bypassed the string and type coercion noise to successfully isolate `1` row where the outer pressure changed dynamically (from `990` to `Null`).

By loading Veridelta's generated **Discrepancy Artifact**, we mathematically verify that the *only* row altered by the pipeline corresponds to the intentional bug fix.

In [ ]:
# Load the isolated discrepancy artifact generated by the Veridelta engine
changed_records: pl.DataFrame = pl.read_parquet("./audit_logs/changed_rows.parquet")

# Assert that 100% of the flagged records match the cross-column physics rule
intentional_fixes = changed_records.filter(  # pyright: ignore[reportUnknownMemberType]
    pl.col("outer_pressure_mb_source") < pl.col("min_pressure_mb_source")
)

if intentional_fixes.height == changed_records.height:
    print(
        "VALIDATION PASSED: Reconciled. All detected changes verify intentional physics bug corrections."
    )
else:
    print("VALIDATION FAILED: Unexpected data regressions detected in the target dataset.")

# Output:
# VALIDATION PASSED: Reconciled. All detected changes verify intentional physics bug corrections.

## 5. Architectural Principle: Separation of Concerns

Veridelta is deliberately designed as an **Inter-System Reconciliation Engine**. Its core utility is comparing System A to System B. Evaluating conditional, cross-column business logic (e.g., `col_A < col_B`) is classified as an **Intra-System** data quality concern.

Veridelta intentionally omits a cross-column Domain Specific Language (DSL) from its configuration specification to ensure:
1. **Security:** Eliminating arbitrary string-evaluation prevents Arbitrary Code Execution (ACE) vectors within production pipelines.
2. **Performance:** The engine avoids AST parsing overhead, relying exclusively on highly optimized native Polars relational algebra.
3. **Maintainability:** Teams are forced to handle business-logic rules upstream within their primary transformation layer (via Polars, dbt, or Great Expectations), restricting Veridelta to its primary function: mathematically verifying the integrity of the migration.

---
*(Housekeeping: Purge local audit files generated during execution)*

In [ ]:
import shutil

shutil.rmtree("./audit_logs", ignore_errors=True)
print("SYSTEM: Audit artifacts purged.")

# Output:
# SYSTEM: Audit artifacts purged.